In [0]:
%run ./01_ga_core



In [0]:
%run ./02_spark_engine


In [0]:
%run ./04_evaluation

In [0]:
from pyspark.sql.functions import when
import pandas as pd
import matplotlib.pyplot as plt
import random

random.seed(42)

# GA settings — defined once so RS uses the exact same budget
CC_POP_SIZE   = 6
CC_GENS       = 6
CC_TOTAL_EVALS = CC_POP_SIZE * CC_GENS   # = 36

HIGGS_POP_SIZE   = 4
HIGGS_GENS       = 4
HIGGS_TOTAL_EVALS = HIGGS_POP_SIZE * HIGGS_GENS  # = 16

# =========================================================
# LOAD CREDIT CARD DATA
# =========================================================
df = spark.read.csv(
    "/Volumes/ga_automl_premium/default/mydata/creditcard.csv",
    header=True,
    inferSchema=True
)

df = df.withColumnRenamed("Class", "label")

# Handle class imbalance via inverse-frequency weighting
counts = df.groupBy("label").count().collect()
count_dict = {r["label"]: r["count"] for r in counts}
total = sum(count_dict.values())

w0 = total / (2 * count_dict[0])
w1 = total / (2 * count_dict[1])

df = df.withColumn("weight", when(df.label == 1, w1).otherwise(w0))
df = df.repartition(8)
df.count()
print("Credit Card Data Loaded")

# =========================================================
# LOAD HIGGS DATA
# =========================================================
df_higgs = spark.read.csv(
    "/Volumes/ga_automl_premium/default/mydata/HIGGS.csv.gz",
    header=False,
    inferSchema=True
)

cols = ["label"] + [f"f{i}" for i in range(1, len(df_higgs.columns))]
df_higgs = df_higgs.toDF(*cols)
# df_higgs = df_higgs.sample(0.1).repartition(8)
# In 00_main_runner.ipynb — change this line
df_higgs = df_higgs.sample(0.05).repartition(8)  # was 0.1 → now 5% = ~550K rows
df_higgs.count()
print("HIGGS Data Loaded")

Credit Card Data Loaded
HIGGS Data Loaded


In [0]:
# =========================================================
# GA ON CREDIT CARD
# =========================================================
print("\n====== RUNNING GA ON CREDIT CARD ======")

best_cc, history_cc, avg_history_cc = run_ga_fast(
    lambda pop: evaluate_population(pop, df, use_pca=False, use_weight=True),
    generations=CC_GENS,
    pop_size=CC_POP_SIZE
)

# =========================================================
# RANDOM SEARCH ON CREDIT CARD  (same budget as GA)
# =========================================================
print("\n====== RUNNING RANDOM SEARCH ON CREDIT CARD ======")
print(f"(Budget: {CC_TOTAL_EVALS} evaluations = {CC_POP_SIZE} pop x {CC_GENS} generations)")

best_cc_rs, best_cc_rs_fitness, history_cc_rs = run_random_search(
    eval_fn=lambda pop: evaluate_population(pop, df, use_pca=False, use_weight=True),
    n_evals=CC_TOTAL_EVALS,
    batch_size=CC_POP_SIZE
)

# =========================================================
# CREDIT CARD: GA vs RANDOM SEARCH COMPARISON PLOT
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: GA convergence (best + average per generation)
axes[0].plot(history_cc,     marker='o', label='GA Best',    color='blue')
axes[0].plot(avg_history_cc, marker='s', label='GA Average', color='lightblue', linestyle='--')
axes[0].set_xlabel("Generation / Batch")
axes[0].set_ylabel("Fitness (0.6*AUC + 0.4*F1)")
axes[0].set_title("Credit Card — GA Convergence")
axes[0].legend()
axes[0].grid(True)

# Right: GA best vs Random Search best-so-far over same number of batches
axes[1].plot(history_cc,    marker='o', label=f'GA (best={max(history_cc):.4f})',       color='blue')
axes[1].plot(history_cc_rs, marker='x', label=f'Random Search (best={best_cc_rs_fitness:.4f})', color='red', linestyle='--')
axes[1].set_xlabel("Generation / Batch")
axes[1].set_ylabel("Best Fitness Found So Far")
axes[1].set_title("Credit Card — GA vs Random Search")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\nCredit Card — GA best     : {max(history_cc):.4f}")
print(f"Credit Card — RS best     : {best_cc_rs_fitness:.4f}")
print(f"GA improvement over RS    : {(max(history_cc) - best_cc_rs_fitness):.4f}")


====== RUNNING GA ON CREDIT CARD ======

=== Generation 0 ===

Evaluating 6 chromosomes...
Preprocessed and cached: 228042 train / 56765 test rows

Model     : lr
Params    : {'maxIter': 10, 'regParam': 0.1, 'elasticNetParam': 0.5}
AUC       : 0.9785
PR-AUC    : 0.6725
Accuracy  : 0.9974
Precision : 0.9987
Recall    : 0.9974
F1 Score  : 0.9979


Model     : gbt
Params    : {'maxIter': 20, 'maxDepth': 3, 'stepSize': 0.05, 'subsamplingRate': 0.8}
AUC       : 0.9886
PR-AUC    : 0.8037
Accuracy  : 0.9993
Precision : 0.9993
Recall    : 0.9993
F1 Score  : 0.9993


Model     : lr
Params    : {'maxIter': 10, 'regParam': 0.01, 'elasticNetParam': 0.0}
AUC       : 0.9793
PR-AUC    : 0.6169
Accuracy  : 0.9876
Precision : 0.9982
Recall    : 0.9876
F1 Score  : 0.9923


Model     : gbt
Params    : {'maxIter': 20, 'maxDepth': 5, 'stepSize': 0.1, 'subsamplingRate': 0.6}
AUC       : 0.9643
PR-AUC    : 0.8165
Accuracy  : 0.9995
Precision : 0.9995
Recall    : 0.9995
F1 Score  : 0.9995


Model     : svm
P

In [0]:
# =========================================================
# GA ON HIGGS
# =========================================================
print("\n====== RUNNING GA ON HIGGS ======")

best_higgs, history_higgs, avg_history_higgs = run_ga_fast(
    lambda pop: evaluate_population(pop, df_higgs, use_pca=True, use_weight=False),
    generations=3,
    pop_size=3
)

# =========================================================
# RANDOM SEARCH ON HIGGS  (same budget as GA: 3 pop x 3 gen = 9 evaluations)
# =========================================================
print("\n====== RUNNING RANDOM SEARCH ON HIGGS ======")
print("(Budget: 9 evaluations = 3 pop x 3 generations)")

best_higgs_rs, best_higgs_rs_fitness, history_higgs_rs = run_random_search(
    eval_fn=lambda pop: evaluate_population(pop, df_higgs, use_pca=True, use_weight=False),
    n_evals=9,       # must match pop_size x generations above
    batch_size=3     # must match pop_size above
)

# =========================================================
# HIGGS: GA vs RANDOM SEARCH COMPARISON PLOT
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_higgs,     marker='o', label='GA Best',    color='green')
axes[0].plot(avg_history_higgs, marker='s', label='GA Average', color='lightgreen', linestyle='--')
axes[0].set_xlabel("Generation / Batch")
axes[0].set_ylabel("Fitness (0.6*AUC + 0.4*F1)")
axes[0].set_title("HIGGS — GA Convergence")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_higgs,    marker='o', label=f'GA (best={max(history_higgs):.4f})',                color='green')
axes[1].plot(history_higgs_rs, marker='x', label=f'Random Search (best={best_higgs_rs_fitness:.4f})', color='orange', linestyle='--')
axes[1].set_xlabel("Generation / Batch")
axes[1].set_ylabel("Best Fitness Found So Far")
axes[1].set_title("HIGGS — GA vs Random Search")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\nHIGGS — GA best        : {max(history_higgs):.4f}")
print(f"HIGGS — RS best        : {best_higgs_rs_fitness:.4f}")
print(f"GA improvement over RS : {(max(history_higgs) - best_higgs_rs_fitness):.4f}")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:465)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:80)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:348)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
# =========================================================
# BEST MODELS — RE-EVALUATE FOR FULL METRICS
# =========================================================
evaluated_cc    = evaluate_population(best_cc, df, use_pca=False, use_weight=True)
best_cc_fitness, best_cc_model = max(evaluated_cc, key=lambda x: x[0])

evaluated_higgs = evaluate_population(best_higgs, df_higgs, use_pca=True, use_weight=False)
best_higgs_fitness, best_higgs_model = max(evaluated_higgs, key=lambda x: x[0])

print("\n====== BEST CREDIT CARD MODEL (GA) ======")
print("Model Type :", best_cc_model[0])
print("Parameters :", best_cc_model[1])
print("Fitness    :", round(best_cc_fitness, 4))

print("\n====== BEST HIGGS MODEL (GA) ======")
print("Model Type :", best_higgs_model[0])
print("Parameters :", best_higgs_model[1])
print("Fitness    :", round(best_higgs_fitness, 4))

# Baseline LR
baseline_cc = run_baseline(df)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8355509062436567>, line 4
      1 # =========================================================
      2 # BEST MODELS — RE-EVALUATE FOR FULL METRICS
      3 # =========================================================
----> 4 evaluated_cc    = evaluate_population(best_cc, df, use_pca=False, use_weight=True)
      5 best_cc_fitness, best_cc_model = max(evaluated_cc, key=lambda x: x[0])
      7 evaluated_higgs = evaluate_population(best_higgs, df_higgs, use_pca=True, use_weight=False)

NameError: name 'best_cc' is not defined

In [0]:
# =========================================================
# SCALABILITY TEST
# =========================================================
print("\n====== SCALABILITY TEST ======")
print("(Using single best model only — 1 model × 4 fractions = 4 jobs)")

scal_results = scalability_test(
    df_higgs,
    lambda d: evaluate_population([best_higgs_model], d, use_pca=True, use_weight=False)
)

scal_df = pd.DataFrame(scal_results, columns=["Data Fraction", "Execution Time"])

plt.figure()
plt.plot(scal_df["Data Fraction"], scal_df["Execution Time"], marker='o')
plt.xlabel("Data Size Fraction")
plt.ylabel("Execution Time (seconds)")
plt.title("Scalability Analysis")
plt.grid()
plt.show()

# =========================================================
# PARALLELISM TEST
# =========================================================
print("\n====== PARALLELISM TEST ======")
print("(Using single best model only — 1 model × 4 partitions = 4 jobs)")

par_results = parallelism_test(
    df,
    lambda d: evaluate_population([best_cc_model], d, use_pca=False, use_weight=True)
)

par_df = pd.DataFrame(par_results, columns=["Partitions", "Execution Time"])

plt.figure()
plt.plot(par_df["Partitions"], par_df["Execution Time"], marker='o')
plt.xlabel("Number of Partitions")
plt.ylabel("Execution Time (seconds)")
plt.title("Parallelism vs Performance")
plt.grid()
plt.show()

# =========================================================
# CLASS IMBALANCE VISUALIZATION
# =========================================================
imbalance_df = df.groupBy("label").count().toPandas()

plt.figure()
plt.bar(imbalance_df["label"].astype(str), imbalance_df["count"])
plt.xlabel("Class Label")
plt.ylabel("Count")
plt.title("Class Distribution (Imbalance)")
plt.show()


====== SCALABILITY TEST ======
(Using single best model only — 1 model × 4 fractions = 4 jobs)


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8355509062436568>, line 7
      4 print("\n====== SCALABILITY TEST ======")
      5 print("(Using single best model only — 1 model × 4 fractions = 4 jobs)")
----> 7 scal_results = scalability_test(
      8     df_higgs,
      9     lambda d: evaluate_population([best_higgs_model], d, use_pca=True, use_weight=False)
     10 )
     12 scal_df = pd.DataFrame(scal_results, columns=["Data Fraction", "Execution Time"])
     14 plt.figure()

File <command-8355509062436572>, line 108, in scalability_test(df, fn)
    105 sample = df.sample(withReplacement=False, fraction=frac, seed=42)
    107 start = time.time()
--> 108 fn(sample)
    109 duration = time.time() - start
    111 print(f"Fraction {frac} -> {duration:.2f} sec")

File <command-8355509062436568>, line 9, in <lambda>(d)
      4 print("\n====== SCALABILITY TEST ======")
 

In [0]:
import pandas as pd

print("\n==================================================")
print("FINAL BEST MODELS WITH METRICS")
print("==================================================")

# ---------------------------
# HELPER: full metrics for one chromosome
# ---------------------------
def get_metrics(chromosome, df, use_pca=False, use_weight=False):
    from pyspark.ml import Pipeline
    from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

    model_type, params = chromosome

    stages = build_pipeline(df, use_pca)
    model  = build_model(model_type, params, use_weight)  # FIX: was incorrectly passing use_pca here

    pipeline = Pipeline(stages=stages + [model])

    train, test = df.randomSplit([0.8, 0.2], seed=42)
    fitted = pipeline.fit(train)
    preds  = fitted.transform(test)

    auc       = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(preds)
    pr_auc    = BinaryClassificationEvaluator(metricName="areaUnderPR").evaluate(preds)
    accuracy  = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(preds)
    precision = MulticlassClassificationEvaluator(metricName="weightedPrecision").evaluate(preds)
    recall    = MulticlassClassificationEvaluator(metricName="weightedRecall").evaluate(preds)
    f1        = MulticlassClassificationEvaluator(metricName="f1").evaluate(preds)

    return {"AUC": auc, "PR_AUC": pr_auc, "Accuracy": accuracy,
            "Precision": precision, "Recall": recall, "F1": f1}


# ---------------------------
# CREDIT CARD
# ---------------------------
cc_metrics    = get_metrics(best_cc_model,    df,       use_pca=False, use_weight=True)
cc_rs_metrics = get_metrics(best_cc_rs,       df,       use_pca=False, use_weight=True)

print("\n====== CREDIT CARD BEST MODEL (GA) ======")
print(f"Model : {best_cc_model[0]}  |  Params : {best_cc_model[1]}")
print(f"Fitness: {best_cc_fitness:.4f}")
for k, v in cc_metrics.items(): print(f"  {k:10}: {v:.4f}")

print("\n====== CREDIT CARD BEST MODEL (Random Search) ======")
print(f"Model : {best_cc_rs[0]}  |  Params : {best_cc_rs[1]}")
print(f"Fitness: {best_cc_rs_fitness:.4f}")
for k, v in cc_rs_metrics.items(): print(f"  {k:10}: {v:.4f}")


# ---------------------------
# HIGGS
# ---------------------------
higgs_metrics    = get_metrics(best_higgs_model, df_higgs, use_pca=True, use_weight=False)
higgs_rs_metrics = get_metrics(best_higgs_rs,    df_higgs, use_pca=True, use_weight=False)

print("\n====== HIGGS BEST MODEL (GA) ======")
print(f"Model : {best_higgs_model[0]}  |  Params : {best_higgs_model[1]}")
print(f"Fitness: {best_higgs_fitness:.4f}")
for k, v in higgs_metrics.items(): print(f"  {k:10}: {v:.4f}")

print("\n====== HIGGS BEST MODEL (Random Search) ======")
print(f"Model : {best_higgs_rs[0]}  |  Params : {best_higgs_rs[1]}")
print(f"Fitness: {best_higgs_rs_fitness:.4f}")
for k, v in higgs_rs_metrics.items(): print(f"  {k:10}: {v:.4f}")


# ---------------------------
# FULL COMPARISON TABLE
# ---------------------------
summary = pd.DataFrame([
    ["Credit Card", "GA",            best_cc_model[0],    best_cc_model[1],    best_cc_fitness,        *cc_metrics.values()],
    ["Credit Card", "Random Search", best_cc_rs[0],       best_cc_rs[1],       best_cc_rs_fitness,     *cc_rs_metrics.values()],
    ["HIGGS",       "GA",            best_higgs_model[0], best_higgs_model[1], best_higgs_fitness,     *higgs_metrics.values()],
    ["HIGGS",       "Random Search", best_higgs_rs[0],    best_higgs_rs[1],    best_higgs_rs_fitness,  *higgs_rs_metrics.values()],
],
columns=[
    "Dataset", "Method", "Model", "Parameters", "Fitness",
    "AUC", "PR_AUC", "Accuracy", "Precision", "Recall", "F1"
])

print("\n==================================================")
print("FULL COMPARISON TABLE")
print("==================================================")
display(summary)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can